In [ ]:
from torch import nn
import torch

class Embedding(nn.Module):
    def __init__(self, vocab_size=30522, dim=768):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, dim)

    def forward(self, x):
        return self.token_embedding(x)

class PositionalEncoding(nn.Module):
    def __init__(self, sql_len=512, dim=768, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # CLIP uses learnable positional embeddings instead of sinusoidal ones
        self.pe = nn.Parameter(torch.zeros(1, sql_len, dim))
        # pe = torch.zeros(1, sql_len, dim)
        # # (1, sql_len, dim)
        # position = torch.arange(0, sql_len).unsqueeze(1)
        # # (sql_len, 1)
        # div_term = 10000 ** (torch.arange(0, dim, 2) / dim)
        # pe[0, :, 0::2] = torch.sin(position/div_term)
        # pe[0, :, 1::2] = torch.cos(position/div_term)


    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :].detach())

class MultiHeadAttention(nn.Module):
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0, "Dimension must be divisible by number of heads."
        self.num_heads = num_heads
        self.w_k = nn.Linear(dim, dim)
        self.w_q = nn.Linear(dim, dim)
        self.w_v = nn.Linear(dim, dim)
        self.w_o = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(q,k,v, dropout=None, mask=None):
        k = k.transpose(-2, -1)
        attention_scores = (q @ k) / torch.sqrt(torch.tensor(q.size(-1), dtype=torch.float32))
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            attention_scores = attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = torch.softmax(attention_scores, dim=-1)
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        return attention_scores @ v

    def forward(self, x, mask=None):
        k = self.w_k(x)
        q = self.w_q(x)
        v = self.w_v(x)

        k = k.view(k.size(0), k.size(1), self.num_heads, -1).transpose(1,2)
        q = q.view(q.size(0), q.size(1), self.num_heads, -1).transpose(1,2)
        v = v.view(v.size(0), v.size(1), self.num_heads, -1).transpose(1,2)

        attention = self.attention(q,k,v, self.dropout, mask)
        attention = attention.transpose(1,2).contiguous().view(x.size(0), x.size(1), -1)
        return self.w_o(attention)

class FeedForward(nn.Module):
    def __init__(self, dim=768, hidden_dim=3072, dropout=0.1):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.sequential(x)

class Normalization(nn.Module):
    def __init__(self, eps=1e-6, dim=768):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        # Formula: y = gamma * (x - mean) / sqrt(variance + epsilon) + beta

    def forward(self, x):
        return self.gamma * (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False) + self.eps) + self.beta

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=30522, dim=768, num_heads=8, hidden_dim=3072, num_layers=12, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.embedding = Embedding(vocab_size, dim)
        self.pos_encoding = PositionalEncoding(max_seq_len, dim, dropout)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "mha": MultiHeadAttention(dim, num_heads, dropout),
                "ffn": FeedForward(dim, hidden_dim, dropout),
                "norm1": Normalization(dim),
                "norm2": Normalization(dim),
            })
            for _ in range(num_layers)
        ])
        self.norm = Normalization(dim)

    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = x + layer["mha"](layer["norm1"](x), mask)
            x = x + layer["ffn"](layer["norm2"](x))
        return self.norm(x[:, -1])